# TADS23 · Lecture 8 · A query, a constraint and rules
The same question, *is a luminaire type below or above 110 lm/W?*, four ways: a SPARQL query, a SHACL-SPARQL constraint, a SHACL rule and the SWRL rules, all on `rules/cpo-rules.ttl`.

**Where:** Google Colab, in your browser; nothing is installed on your computer. Open it from the CPO repository: [open in Colab](https://colab.research.google.com/github/RichZele/cpo/blob/main/notebooks/cpo-lecture8-rules.ipynb), then **File › Save a copy in Drive**. Run the cells one at a time, when I say the number.

## ► Run 1 · Install
Java for the reasoner Pellet; owlready2 0.47 (newer versions bring a Pellet that needs Java 25); pySHACL, which brings rdflib. About a minute.

In [ ]:
# ► Run 1 · Install Java (for Pellet), owlready2 and pySHACL
!apt-get -qq install -y default-jre-headless > /dev/null
!pip install -q owlready2==0.47 pyshacl
!java -version

## ► Run 2 · The files from the CPO repository

In [ ]:
# ► Run 2 · Download the data, the shapes, the rule and the query
import urllib.request, rdflib
BASE = 'https://raw.githubusercontent.com/RichZele/cpo/main/'
for f in ['rules/cpo-rules.ttl', 'shapes/cpo-efficacy-sparql-shape.ttl',
          'shapes/cpo-derive-type-rule.ttl', 'queries/cpo-below-minimum-efficacy.rq']:
    urllib.request.urlretrieve(BASE + f, f.split('/')[-1])
data = rdflib.Graph().parse('cpo-rules.ttl')
print('data graph:', len(data), 'triples')      # 779

## ► Run 3 · A SPARQL query: which types are below 110 lm/W?
The answer is a table of rows. Nothing is checked or changed.

In [ ]:
# ► Run 3 · An ordinary SELECT on the data graph
for row in data.query(open('cpo-below-minimum-efficacy.rq').read()):
    print(row.type, row.flux, row.power)      # the Pleiad: 938, 9.5

## ► Run 4 · A SHACL-SPARQL constraint: the same condition, as a check
The answer is a validation report: one result per row the query returns.

In [ ]:
# ► Run 4 · Validate with the SPARQL-based constraint
import pyshacl
shapes = rdflib.Graph().parse('cpo-efficacy-sparql-shape.ttl')
conforms, report_graph, report_text = pyshacl.validate(data, shacl_graph=shapes, inference='none')
print('Conforms:', conforms)
print(report_text)                               # one violation: Efficacy 99 lm/W is below 110 lm/W.

## ► Run 5 · A SHACL rule: derive, instead of report
`sh:rule` is from SHACL Advanced Features, not SHACL Core: pySHACL runs it only with `advanced=True`. We work on a copy, so `data` stays as it is.

In [ ]:
# ► Run 5 · Run the SHACL rule on a copy of the data
from rdflib import RDF, Namespace
CPO = Namespace('https://w3id.org/cpo#')
work = rdflib.Graph() + data
rule = rdflib.Graph().parse('cpo-derive-type-rule.ttl')
pyshacl.validate(work, shacl_graph=rule, advanced=True, inplace=True)
print(len(work) - len(data), 'new triples')      # 2
for t in work.subjects(RDF.type, CPO.HighEfficacyLuminaireType):
    print(t)

## ► Run 6 · The SWRL rules: load them with owlready2
owlready2 reads RDF/XML or N-Triples, not Turtle, so rdflib converts the file first.

In [ ]:
# ► Run 6 · Load the ontology and its two SWRL rules
from pathlib import Path
from rdflib import Graph
from owlready2 import World, sync_reasoner_pellet
root = Path('.').resolve()
file = root / 'lighting.nt'
Graph().parse(root / 'cpo-rules.ttl').serialize(file, format='nt')
world = World()
ontology = world.get_ontology(file.as_posix()).load()
cpo = ontology.get_namespace('https://w3id.org/cpo#')
for rule in world.rules():
    print(rule)                                  # R1 and R2

## ► Run 7 · Run Pellet: OWL axioms and SWRL rules together

In [ ]:
# ► Run 7 · Reason with Pellet, then read the inferred instances
with ontology:
    sync_reasoner_pellet(world, infer_property_values=True, infer_data_property_values=True)
for cls in [cpo.HighEfficacyLuminaireType, cpo.HighEfficacyLuminaire]:
    items = list(cls.instances())
    print(cls.name, len(items), sorted(i.name for i in items))   # 2 types, 23 luminaires